# AI Cycling Coach — GPU Training (Kaggle)

**Settings (right sidebar) → Accelerator → GPU T4 x2** before running.

Then click **Run All**. No uploads needed — generates data here (~3 min) then trains on GPU (~1–2 h).

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Settings → Accelerator → GPU T4 x2')

In [ ]:
# ── 2. Clone repo + set paths ────────────────────────────────────────────────
import os, sys

REPO    = 'https://github.com/yossibello/ai-coach.git'
WORKDIR = '/kaggle/working/ai-coach'

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
else:
    !cd {WORKDIR} && git pull

%cd {WORKDIR}

for p in [f'{WORKDIR}/backend', WORKDIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ['PYTHONPATH']      = f'{WORKDIR}/backend'
os.environ['PYTHONIOENCODING'] = 'utf-8'

!mkdir -p ml/data backend/models
print('cwd:', os.getcwd())
print('sys.path[0:3]:', sys.path[:3])

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
!pip install pyarrow --upgrade -q
import torch, pandas
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('pandas:', pandas.__version__)

In [ ]:

# ── 4. Load training data ────────────────────────────────────────────────────
# TWO OPTIONS — set MODE below:
#
#   'dataset'  → you added the ai-coach-synthetic Kaggle dataset (recommended, instant)
#   'generate' → generate here in Kaggle (~3 min for 20K, ~8 min for 50K)

import os, sys, subprocess, multiprocessing, pandas as pd

MODE          = 'generate'  # ← 'dataset' | 'generate'
ATHLETES      = 50_000      # only used when MODE='generate'
DATASET_PATH  = '/kaggle/input/ai-coach-synthetic/synthetic.parquet'
DATA_FILE     = 'ml/data/synthetic.parquet'

os.makedirs('ml/data', exist_ok=True)

if MODE == 'dataset':
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_PATH}\n'
            'Add it: right panel → Add data → search "ai-coach-synthetic"'
        )
    print(f'Using Kaggle dataset: {DATASET_PATH}  ({os.path.getsize(DATASET_PATH)/1e6:.0f} MB)')
    DATA_FILE = DATASET_PATH

elif MODE == 'generate':
    workers = max(1, multiprocessing.cpu_count() - 1)
    print(f'Generating {ATHLETES:,} athletes using {workers} workers…')

    # Use subprocess so we capture stdout+stderr and get a real exception on failure.
    # The !shell magic swallows errors and leaves DATA_FILE missing.
    env = os.environ.copy()
    env['PYTHONPATH'] = f"{os.getcwd()}/backend:{os.getcwd()}"
    result = subprocess.run(
        [sys.executable, '-m', 'ml.training.generate_synthetic',
         '--athletes', str(ATHLETES),
         '--workers',  str(workers),
         '--output',   DATA_FILE],
        env=env,
        capture_output=False,   # stream output live to the cell
    )
    if result.returncode != 0:
        raise RuntimeError(
            f'generate_synthetic failed (exit {result.returncode}).\n'
            'Scroll up for the traceback printed above.'
        )

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Expected {DATA_FILE} but it was not created.\n'
        'Check the output above for errors.'
    )

df = pd.read_parquet(DATA_FILE)
assert 'risk_ot_class'   in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'risk_inj_target' in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'pc_5s_wkg'       in df.columns, 'Old parquet — missing power curve features, regenerate'
print(f'✓ Data ready: {len(df):,} rows, {df.athlete_id.nunique():,} athletes, {len(df.columns)} cols')
print(f'  Power curve columns present: pc_5s_wkg={df.pc_5s_wkg.notna().mean()*100:.0f}% non-null')
del df


In [ ]:

# ── 5. Train ─────────────────────────────────────────────────────────────────
import sys, os, torch, argparse, multiprocessing

n_gpus  = torch.cuda.device_count() if torch.cuda.is_available() else 1
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

if   vram_gb >= 45: per_gpu = 4096
elif vram_gb >= 38: per_gpu = 2048
elif vram_gb >= 20: per_gpu = 1024
else:               per_gpu = 1024

BATCH_SIZE      = per_gpu * n_gpus
STEPS_PER_EPOCH = 5000
EPOCHS          = 50
MODEL_FILE      = 'backend/models/cycling_coach.pt'
dl_workers      = min(6, multiprocessing.cpu_count())
os.environ['DATALOADER_WORKERS'] = str(dl_workers)

# 3e-4: stable for 8M params + sqrt class weights. (6e-4 caused oscillation
# because class weights amplify gradients on rare classes up to 3×.)
LEARNING_RATE = 3e-4

# ── Checkpoint / resume logic ─────────────────────────────────────────────────
FORCE_FRESH = False  # ← set False to auto-resume from last saved checkpoint

if FORCE_FRESH:
    if os.path.exists(MODEL_FILE):
        os.remove(MODEL_FILE)
        print('*** FORCE_FRESH: deleted old checkpoint, starting from epoch 1 ***')
    else:
        print('*** Starting fresh training from epoch 1 ***')
    CHECKPOINT = None
else:
    CHECKPOINT = MODEL_FILE if os.path.exists(MODEL_FILE) else None
    if CHECKPOINT:
        _meta = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
        _ep   = _meta.get('metrics', {}).get('epoch', '?') if isinstance(_meta, dict) else '?'
        print(f'*** Resuming from checkpoint: epoch {_ep} → will train epochs {_ep+1 if isinstance(_ep,int) else "?"}-{EPOCHS} ***')
    else:
        print('*** Starting fresh training from epoch 1 ***')

print(f'GPUs: {n_gpus}  |  VRAM/GPU: {vram_gb:.1f} GB  |  batch: {BATCH_SIZE} ({per_gpu}/GPU)')
print(f'Steps/epoch: {STEPS_PER_EPOCH}  |  LR: {LEARNING_RATE}  |  Workers: {dl_workers}')
print(f'Data: {DATA_FILE}  ({os.path.getsize(DATA_FILE)/1e6:.0f} MB)')
print('─' * 60)

for _k in list(sys.modules.keys()):
    if 'training.train' in _k:
        del sys.modules[_k]

from ml.training.train import train as run_training
os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)

args = argparse.Namespace(
    data             = DATA_FILE,
    output           = MODEL_FILE,
    checkpoint       = CHECKPOINT,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    steps_per_epoch  = STEPS_PER_EPOCH,
    lr               = LEARNING_RATE,
    seq_len          = 90,
    val_frac         = 0.1,
    seed             = 42,
    d_model          = 256,
    nhead            = 8,
    num_layers       = 8,
    d_ff             = 1024,
    dropout          = 0.15,
    fast             = False,
    patience         = 20,
    compile          = False,
    no_amp           = False,
    mask_workout_type = False,  # synthetic data: workout_type is ground-truth, keep it
)

run_training(args)
print(f'\n✓ Training complete! Model → {MODEL_FILE}')


In [ ]:
# ── 6. Copy model to /kaggle/working/ so Kaggle saves it as output ────────────
import shutil, os

OUT = '/kaggle/working/cycling_coach.pt'
shutil.copy(MODEL_FILE, OUT)
print(f'✓ Model saved to {OUT}')
print('  → After the notebook finishes, go to the Output tab and download it.')

In [ ]:

# ── 6b. (Optional) Fine-tune on GoldenCheetah real data ─────────────────────
# To enable: right panel → Add data → search "goldencheetah opendata"
# Then set FINETUNE = True below.

FINETUNE         = False   # ← flip to True after adding the dataset
GC_INPUT_DIR     = '/kaggle/input/goldencheetah-opendata-athlete-activity-and-mmp'
GC_PARQUET       = 'ml/data/goldencheetah.parquet'
FINETUNE_EPOCHS  = 15
FINETUNE_LR      = 5e-5   # 10× lower than pre-training (fine-tune conservatively)

if FINETUNE:
    import os, sys, argparse, subprocess

    gc_exists = os.path.isdir(GC_INPUT_DIR)
    if not gc_exists:
        print('⚠ GoldenCheetah dataset not found at', GC_INPUT_DIR)
        print('  Add it: right panel → Add data → search "goldencheetah opendata"')
    else:
        # ── Convert GoldenCheetah → our parquet schema ──────────────────────
        if not os.path.exists(GC_PARQUET):
            print('Converting GoldenCheetah data…')
            env = os.environ.copy()
            env['PYTHONPATH'] = f"{os.getcwd()}/backend:{os.getcwd()}"
            result = subprocess.run(
                [sys.executable, '-m', 'ml.training.convert_goldencheetah',
                 '--input',  GC_INPUT_DIR,
                 '--output', GC_PARQUET],
                env=env, capture_output=False,
            )
            if result.returncode != 0:
                raise RuntimeError('convert_goldencheetah failed — see output above')
        else:
            print(f'Using existing {GC_PARQUET}')

        import pandas as pd
        gc_df = pd.read_parquet(GC_PARQUET)
        print(f'GoldenCheetah: {len(gc_df):,} rows, {gc_df["athlete_id"].nunique()} athletes')

        # ── Fine-tune: load pre-trained checkpoint, train on real data ──────
        print('\nFine-tuning on real data…')
        from ml.training.train import train as run_training

        ft_args = argparse.Namespace(
            data              = GC_PARQUET,
            output            = MODEL_FILE,
            checkpoint        = MODEL_FILE,   # start from pre-trained weights
            epochs            = FINETUNE_EPOCHS,
            batch_size        = BATCH_SIZE,
            steps_per_epoch   = None,         # use all batches (dataset is small)
            lr                = FINETUNE_LR,
            seq_len           = 90,
            val_frac          = 0.15,
            seed              = 42,
            d_model           = 256,
            nhead             = 8,
            num_layers        = 8,
            d_ff              = 1024,
            dropout           = 0.1,
            fast              = False,
            patience          = 8,
            compile           = False,
            no_amp            = False,
            mask_workout_type = True,   # real data: workout_type is inferred, zero it out
        )
        run_training(ft_args)
        print(f'✓ Fine-tuning complete! Model → {MODEL_FILE}')
else:
    print('Fine-tuning skipped (FINETUNE=False). Set True and add the GoldenCheetah dataset to enable.')


In [ ]:
# ── 7. (Optional) Push model to GitHub ──────────────────────────────────────
# Create a PAT at https://github.com/settings/tokens (Classic, repo scope)
# then paste it when prompted.

from getpass import getpass
token = getpass('GitHub PAT (hidden): ')

!git config user.email 'kaggle@training'
!git config user.name  'Kaggle Training'
!git remote set-url origin https://{token}@github.com/yossibello/ai-coach.git
!cp /kaggle/working/cycling_coach.pt backend/models/cycling_coach.pt
!git add backend/models/cycling_coach.pt
!git commit -m "Trained model: {ATHLETES} athletes, {EPOCHS} epochs (Kaggle GPU)"
!git push origin main
print('✓ Model pushed to GitHub!')

In [ ]:
# ── 8. Sanity check ──────────────────────────────────────────────────────────
import torch, sys
from app.ml.model import CyclingTransformer

ckpt = torch.load(MODEL_FILE, map_location='cpu')
cfg  = ckpt.get('config', {})
m    = CyclingTransformer(
    d_model=cfg.get('d_model', 128),
    nhead=cfg.get('nhead', 8),
    num_layers=cfg.get('num_layers', 6),
    dim_feedforward=cfg.get('dim_feedforward', 512),
)
m.load_state_dict(ckpt['state_dict'])
m.eval()
print('Model loaded OK')
print('Params:', sum(p.numel() for p in m.parameters()))
print('Best val loss:', ckpt.get('metrics', {}).get('val_loss', 'n/a'))
print('Epoch:',        ckpt.get('metrics', {}).get('epoch',    'n/a'))